# Parameter Sweep: Racetrack MRR DWDM Demultiplexer

This notebook sweeps the key design parameters of the racetrack MRR —
coupling gap $g$, coupling length $L_c$, and bend radius $R$ —
and evaluates the impact on insertion loss (IL), extinction ratio (ER),
quality factor $Q$, and shape factor $\eta$.

**Usage:** Run all cells top-to-bottom, or modify the sweep ranges below.

In [ ]:
import sys
from pathlib import Path

# Add project paths
repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root / 'design'))
sys.path.insert(0, str(repo_root / 'simulation'))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import config as cfg
from spectral_response import ring_spectrum, extract_metrics

print('Setup complete.')

## 1. Sweep κ² (Power Coupling Coefficient)

In [ ]:
wl = np.linspace(1540, 1560, 2000)   # wavelength array [nm]
target_nm = 1550.0
lc_nom = cfg.COUPLING_LEN

kappa_values = np.linspace(0.02, 0.30, 15)

records = []
for kappa in kappa_values:
    T_thru, T_drop = ring_spectrum(wl, lc=lc_nom, kappa_sq=kappa)
    m = extract_metrics(wl, T_drop, T_thru)
    records.append({'kappa_sq': round(kappa, 4), **m})

df_kappa = pd.DataFrame(records)
display(df_kappa[['kappa_sq', 'il_db', 'er_db', 'q_factor', 'fwhm_nm', 'shape_factor']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(df_kappa['kappa_sq'], df_kappa['il_db'], 'b-o', ms=4)
axes[0].set_xlabel('κ²')
axes[0].set_ylabel('Insertion Loss (dB)')
axes[0].set_title('IL vs κ²')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_kappa['kappa_sq'], df_kappa['er_db'], 'r-o', ms=4)
axes[1].set_xlabel('κ²')
axes[1].set_ylabel('Extinction Ratio (dB)')
axes[1].set_title('ER vs κ²')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df_kappa['kappa_sq'], df_kappa['q_factor'], 'g-o', ms=4)
axes[2].set_xlabel('κ²')
axes[2].set_ylabel('Quality Factor Q')
axes[2].set_title('Q vs κ²')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../latex/figures/sweep_kappa.png', dpi=150)
plt.show()
print('Figure saved.')

## 2. Sweep Bend Radius $R$

In [ ]:
import math

radius_values = np.arange(5, 25, 2)   # 5–23 um

records_r = []
for R in radius_values:
    # Recompute coupling length to keep resonance at 1550 nm
    circ = 2 * math.pi * R
    nominal_lrt = 2 * cfg.COUPLING_LEN + 2 * math.pi * cfg.BEND_RADIUS
    m = round(cfg.N_EFF * nominal_lrt / (target_nm * 1e-3))
    lrt = m * (target_nm * 1e-3) / cfg.N_EFF
    lc = max((lrt - circ) / 2.0, 1.0)

    T_thru, T_drop = ring_spectrum(wl, lc=lc, kappa_sq=0.08)
    m_dict = extract_metrics(wl, T_drop, T_thru)
    records_r.append({'radius_um': R, 'lc_um': round(lc, 3),
                       'lrt_um': round(lrt * 1e3, 1), **m_dict})

df_radius = pd.DataFrame(records_r)
display(df_radius[['radius_um', 'lc_um', 'lrt_um', 'il_db', 'er_db', 'fsr_nm', 'q_factor']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(df_radius['radius_um'], df_radius['fsr_nm'], 'm-o', ms=4)
axes[0].set_xlabel('Bend Radius (μm)')
axes[0].set_ylabel('FSR (nm)')
axes[0].set_title('FSR vs Bend Radius')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_radius['radius_um'], df_radius['q_factor'], 'c-o', ms=4)
axes[1].set_xlabel('Bend Radius (μm)')
axes[1].set_ylabel('Q Factor')
axes[1].set_title('Q Factor vs Bend Radius')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../latex/figures/sweep_radius.png', dpi=150)
plt.show()

## 3. Sweep Propagation Loss

In [ ]:
loss_values = np.linspace(0.5, 10.0, 20)  # dB/cm

records_loss = []
for loss in loss_values:
    T_thru, T_drop = ring_spectrum(wl, lc=lc_nom, kappa_sq=0.08,
                                    loss_db_per_cm=loss)
    m = extract_metrics(wl, T_drop, T_thru)
    records_loss.append({'loss_db_cm': round(loss, 2), **m})

df_loss = pd.DataFrame(records_loss)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(df_loss['loss_db_cm'], df_loss['il_db'], 'b-o', ms=4)
axes[0].set_xlabel('Propagation loss (dB/cm)')
axes[0].set_ylabel('Drop IL (dB)')
axes[0].set_title('Insertion Loss vs Waveguide Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_loss['loss_db_cm'], df_loss['er_db'], 'r-o', ms=4)
axes[1].set_xlabel('Propagation loss (dB/cm)')
axes[1].set_ylabel('ER (dB)')
axes[1].set_title('Extinction Ratio vs Waveguide Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../latex/figures/sweep_loss.png', dpi=150)
plt.show()

## 4. Summary Table (All Sweeps)

In [ ]:
print('=== κ² Sweep ===')
display(df_kappa)
print('\n=== Radius Sweep ===')
display(df_radius)
print('\n=== Loss Sweep ===')
display(df_loss)